# Uno Feature Regression, Enhanced Version

This notebook extends `uno_feature_regression.ipynb` with richer line-profile features, weighted regression loss, early stopping, and test-only diagnostics for the coupled-coefficient inverse problem.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone or Pull Latest GitHub Code

In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/DrYGuo/Aberration-Simulation.git"
ROOT = Path("/content/Aberration-Simulation")

if ROOT.exists():
    print("Repository already exists. Pulling latest main...")
    subprocess.run(["git", "fetch", "origin", "main"], cwd=ROOT, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=ROOT, check=True)
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
importlib.invalidate_caches()
print("repo root:", ROOT)
print("commit:", commit)

## 3. Install and Verify Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if importlib.util.find_spec("cupy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"], check=True)
    raise RuntimeError("Installed CuPy. Restart the Colab runtime, then rerun from the top.")
if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)

print("Dependencies are ready.")

## 4. Imports

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import csv
import hashlib
import importlib
import json
import math
import platform
import subprocess
import shutil
import sys
import zipfile

import matplotlib.pyplot as plt
import numpy as np

from aberration_simulation.backend import HAS_CUPY, asnumpy, xp
from aberration_simulation.line_profiles import extract_line_profiles_from_stack
from aberration_simulation.optics import SimulationConfig, run_simulation
from aberration_simulation.uno_conventions import (
    add_complex_columns,
    compute_line_characteristics,
    compute_uno_values,
    select_under_over_pairs,
)

sys.modules.pop("feature_regression_model", None)
importlib.invalidate_caches()
from feature_regression_model import (
    FEATURE_COLUMNS as UNO_FEATURE_COLUMNS,
    HARMONIC_TARGETS,
    TARGET_COLUMNS,
    Standardizer,
    target_from_row,
    train_test_split,
)

assert HAS_CUPY, "CuPy is not active. Choose a GPU runtime, restart, and rerun."
print("CuPy GPU path is active.")
print("device count:", xp.cuda.runtime.getDeviceCount())
print("output notebook: enhanced feature regression")


## 5. Generate Training Parameter Cases

In [ ]:
OUTPUT_ROOT = ROOT / "training_results" / "feature_regression_enhanced"
RUN_NAME = "enhanced_coupled10k_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")
OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("run name:", RUN_NAME)
print("output dir:", OUTPUT_DIR)

UNDER_FOCUS_C1_OFFSET = -909
OVER_FOCUS_C1_OFFSET = 909
C1_OFFSETS = [UNDER_FOCUS_C1_OFFSET, OVER_FOCUS_C1_OFFSET]
PROFILE_RADIUS_PIXELS = 80
PROFILE_STEP_DEGREES = 10
NUM_PROFILE_LINES = int(180 / PROFILE_STEP_DEGREES) + 1

BASELINE_PARAMETERS = {
    "C1_offset": 0,
    "A3_amp": 0,
    "S3_amp": 0,
    "A2_amp": 0,
    "B2_amp": 0,
    "C1": 0,
    "C3": 0,
    "A1_amp": 0,
    "A1_phase": 0,
    "A2_phase": 0,
    "A3_phase": 0,
    "S3_phase": 0,
    "B2_phase": 0,
}

COMBINATION_FIELDS = (
    "sweep_label",
    "C1",
    "A1_amp",
    "A1_phase",
    "A2_amp",
    "A2_phase",
    "B2_amp",
    "B2_phase",
    "A3_amp",
    "A3_phase",
    "S3_amp",
    "S3_phase",
    "C3",
)

SCALAR_SWEEP_SPECS = [
    {"label": "C1", "input_field": "C1", "input_values": [-80, -40, -20, 20, 40, 80]},
    {"label": "C3", "input_field": "C3", "input_values": [0.1, 0.2, 0.5, 1.0, 1.5, 2.0]},
]

HARMONIC_SWEEP_SPECS = [
    {"label": "A1", "amp_field": "A1_amp", "phase_field": "A1_phase", "amp_values": [5, 15, 30, 60], "phase_values": [0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5]},
    {"label": "B2/C21", "amp_field": "B2_amp", "phase_field": "B2_phase", "amp_values": [0.5, 1.0, 2.0, 3.0], "phase_values": [0, 45, 90, 135, 180, 225, 270, 315]},
    {"label": "A2", "amp_field": "A2_amp", "phase_field": "A2_phase", "amp_values": [1, 4, 8, 16], "phase_values": [0, 30, 60, 90, 120]},
    {"label": "A3", "amp_field": "A3_amp", "phase_field": "A3_phase", "amp_values": [1, 4, 8, 16], "phase_values": [0, 22.5, 45, 67.5, 90]},
    {"label": "S3/C32", "amp_field": "S3_amp", "phase_field": "S3_phase", "amp_values": [1, 4, 8, 16], "phase_values": [0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5, 180]},
]

WIDE_A3_S3_AMP_VALUES = [0, 1, 4, 8, 16, 25, 50, 75, 100]
WIDE_A3_PHASE_VALUES = [0, 22.5, 45, 67.5, 90]
WIDE_S3_PHASE_VALUES = [0, 45, 90, 135, 180]

C1_A1_C3_GRID_LABEL = "C1_A1_C3_grid"
C1_A1_C3_C1_VALUES = [-40, 0, 40]
C1_A1_C3_C3_VALUES = [0, 0.5, 1.5]
C1_A1_C3_A1_AMP_VALUES = [0, 30, 60]
C1_A1_C3_A1_PHASE_VALUES = [0, 45, 90, 135]

A1_B2_S3_GRID_LABEL = "A1_B2_S3_grid"
A1_B2_S3_A1_AMP_VALUES = [0, 60]
A1_B2_S3_A1_PHASE_VALUES = [0, 90]
A1_B2_S3_B2_AMP_VALUES = [0, 3.0]
A1_B2_S3_B2_PHASE_VALUES = [0, 180]
A1_B2_S3_S3_AMP_VALUES = [0, 100]
A1_B2_S3_S3_PHASE_VALUES = [0, 90, 180]

base_cases = []
for spec in SCALAR_SWEEP_SPECS:
    for value in spec["input_values"]:
        params = dict(BASELINE_PARAMETERS)
        params["sweep_label"] = spec["label"]
        params[spec["input_field"]] = value
        base_cases.append(params)

for spec in HARMONIC_SWEEP_SPECS:
    for amp in spec["amp_values"]:
        for phase in spec["phase_values"]:
            params = dict(BASELINE_PARAMETERS)
            params["sweep_label"] = spec["label"]
            params[spec["amp_field"]] = amp
            params[spec["phase_field"]] = phase
            base_cases.append(params)

for amp in WIDE_A3_S3_AMP_VALUES:
    for phase in WIDE_A3_PHASE_VALUES:
        params = dict(BASELINE_PARAMETERS)
        params["sweep_label"] = "A3_wide"
        params["A3_amp"] = amp
        params["A3_phase"] = phase
        base_cases.append(params)
    for phase in WIDE_S3_PHASE_VALUES:
        params = dict(BASELINE_PARAMETERS)
        params["sweep_label"] = "S3_C32_wide"
        params["S3_amp"] = amp
        params["S3_phase"] = phase
        base_cases.append(params)

for c1_value in C1_A1_C3_C1_VALUES:
    for c3_value in C1_A1_C3_C3_VALUES:
        for a1_amp in C1_A1_C3_A1_AMP_VALUES:
            for a1_phase in C1_A1_C3_A1_PHASE_VALUES:
                params = dict(BASELINE_PARAMETERS)
                params["sweep_label"] = C1_A1_C3_GRID_LABEL
                params["C1"] = c1_value
                params["C3"] = c3_value
                params["A1_amp"] = a1_amp
                params["A1_phase"] = a1_phase
                base_cases.append(params)

for a1_amp in A1_B2_S3_A1_AMP_VALUES:
    for a1_phase in A1_B2_S3_A1_PHASE_VALUES:
        for b2_amp in A1_B2_S3_B2_AMP_VALUES:
            for b2_phase in A1_B2_S3_B2_PHASE_VALUES:
                for s3_amp in A1_B2_S3_S3_AMP_VALUES:
                    for s3_phase in A1_B2_S3_S3_PHASE_VALUES:
                        params = dict(BASELINE_PARAMETERS)
                        params["sweep_label"] = A1_B2_S3_GRID_LABEL
                        params["A1_amp"] = a1_amp
                        params["A1_phase"] = a1_phase
                        params["B2_amp"] = b2_amp
                        params["B2_phase"] = b2_phase
                        params["S3_amp"] = s3_amp
                        params["S3_phase"] = s3_phase
                        base_cases.append(params)

RANDOM_SEED = 19
RANDOM_COUPLED_CASES = 10000
RANDOM_CASE_COUNTS = {
    "coupled_full_random": 2500,
    "coupled_C1_C3_A2_random": 1500,
    "coupled_C1_A1_C3_A2_random": 1500,
    "coupled_A1_A2_B2_random": 1500,
    "coupled_A1_B2_S3_random": 1500,
    "coupled_A3_S3_random": 1000,
    "coupled_sparse_random": 500,
}
assert sum(RANDOM_CASE_COUNTS.values()) == RANDOM_COUPLED_CASES

rng = np.random.default_rng(RANDOM_SEED)

def _uniform(low, high):
    return float(rng.uniform(low, high))

def _maybe_amp(max_value, active_probability=0.85):
    if rng.random() > active_probability:
        return 0.0
    return _uniform(0.0, max_value)

def _randomize(params, fields):
    if "C1" in fields:
        params["C1"] = _uniform(-80.0, 80.0)
    if "C3" in fields:
        params["C3"] = _uniform(0.0, 2.0)
    if "A1" in fields:
        params["A1_amp"] = _maybe_amp(60.0)
        params["A1_phase"] = _uniform(0.0, 180.0)
    if "A2" in fields:
        params["A2_amp"] = _maybe_amp(16.0)
        params["A2_phase"] = _uniform(0.0, 120.0)
    if "B2" in fields:
        params["B2_amp"] = _maybe_amp(3.0)
        params["B2_phase"] = _uniform(0.0, 360.0)
    if "A3" in fields:
        params["A3_amp"] = _maybe_amp(100.0)
        params["A3_phase"] = _uniform(0.0, 90.0)
    if "S3" in fields:
        params["S3_amp"] = _maybe_amp(100.0)
        params["S3_phase"] = _uniform(0.0, 180.0)
    return params

def _append_random_cases(label, count, fields):
    for _ in range(count):
        params = dict(BASELINE_PARAMETERS)
        params["sweep_label"] = label
        base_cases.append(_randomize(params, fields))

_append_random_cases(
    "coupled_full_random",
    RANDOM_CASE_COUNTS["coupled_full_random"],
    ("C1", "C3", "A1", "A2", "B2", "A3", "S3"),
)
_append_random_cases(
    "coupled_C1_C3_A2_random",
    RANDOM_CASE_COUNTS["coupled_C1_C3_A2_random"],
    ("C1", "C3", "A2"),
)
_append_random_cases(
    "coupled_C1_A1_C3_A2_random",
    RANDOM_CASE_COUNTS["coupled_C1_A1_C3_A2_random"],
    ("C1", "C3", "A1", "A2"),
)
_append_random_cases(
    "coupled_A1_A2_B2_random",
    RANDOM_CASE_COUNTS["coupled_A1_A2_B2_random"],
    ("A1", "A2", "B2"),
)
_append_random_cases(
    "coupled_A1_B2_S3_random",
    RANDOM_CASE_COUNTS["coupled_A1_B2_S3_random"],
    ("A1", "B2", "S3"),
)
_append_random_cases(
    "coupled_A3_S3_random",
    RANDOM_CASE_COUNTS["coupled_A3_S3_random"],
    ("A3", "S3"),
)

ALL_FIELDS = ("C1", "C3", "A1", "A2", "B2", "A3", "S3")
for _ in range(RANDOM_CASE_COUNTS["coupled_sparse_random"]):
    active_count = int(rng.integers(2, len(ALL_FIELDS) + 1))
    fields = tuple(rng.choice(ALL_FIELDS, size=active_count, replace=False))
    params = dict(BASELINE_PARAMETERS)
    params["sweep_label"] = "coupled_sparse_random"
    base_cases.append(_randomize(params, fields))

parameters = []
for base_case in base_cases:
    for c1_offset in C1_OFFSETS:
        params = dict(base_case)
        params["C1_offset"] = c1_offset
        parameters.append(params)

print("base cases:", len(base_cases))
print("simulated probe images, including C1 offsets:", len(parameters))
for label in sorted({case["sweep_label"] for case in base_cases}):
    print(f"  {label}: {sum(case['sweep_label'] == label for case in base_cases)} base cases")


## 6. Configure Batched Probe Simulation


In [ ]:
config = SimulationConfig(
    pix_dim=(256, 256),
    real_dim=(1280, 1280),
    app=30,
    sigma=2,
)

# Keep GPU computation vectorized within each batch, but avoid holding all probes
# and FFT temporaries for the full training grid at once.
BATCH_BASE_CASES = 64
BATCH_PROBE_IMAGES = BATCH_BASE_CASES * len(C1_OFFSETS)
ctf_bytes = np.prod(config.pix_dim) * BATCH_PROBE_IMAGES * np.dtype(np.complex128).itemsize
print("batch base cases:", BATCH_BASE_CASES)
print("batch probe images, including C1 offsets:", BATCH_PROBE_IMAGES)
print("approx CTF tensor per batch, GiB:", ctf_bytes / 1024**3)
print("total base cases to process:", len(base_cases))


## 7. Compute Enhanced Feature Values and Save Training Table

In [ ]:
# The baseline 12 Uno features are kept. Extra features expose the raw under- and
# over-focus line-characteristic harmonics before the Uno formulas collapse them.
CHARACTERISTIC_NAMES = ("Xigma", "Mu", "Rho")
HARMONIC_FEATURE_ORDERS = (1, 2, 3, 4)
FOCUS_NAMES = ("under", "over")

EXTRA_FEATURE_COLUMNS = []
for focus_name in FOCUS_NAMES:
    for char_name in CHARACTERISTIC_NAMES:
        EXTRA_FEATURE_COLUMNS.append(f"{focus_name}_{char_name}_mean")
        for order in HARMONIC_FEATURE_ORDERS:
            EXTRA_FEATURE_COLUMNS.append(f"{focus_name}_{char_name}_h{order}_real")
            EXTRA_FEATURE_COLUMNS.append(f"{focus_name}_{char_name}_h{order}_imag")

FEATURE_COLUMNS_ENHANCED = list(UNO_FEATURE_COLUMNS) + EXTRA_FEATURE_COLUMNS
print("Uno features:", len(UNO_FEATURE_COLUMNS))
print("extra raw line-characteristic features:", len(EXTRA_FEATURE_COLUMNS))
print("enhanced total features:", len(FEATURE_COLUMNS_ENHANCED))

def scalar_value(value):
    value_np = asnumpy(value)
    if isinstance(value_np, np.ndarray):
        return value_np.item() if value_np.shape == () else value_np
    return value_np

def add_raw_harmonic_features(row, prefix, chars, angles_deg):
    theta = xp.deg2rad(xp.asarray(angles_deg, dtype=float))
    for char_name in CHARACTERISTIC_NAMES:
        values = chars[char_name]
        row[f"{prefix}_{char_name}_mean"] = float(scalar_value(xp.mean(values)))
        for order in HARMONIC_FEATURE_ORDERS:
            harmonic = 2 * xp.mean(values * xp.exp(1j * order * theta))
            harmonic = scalar_value(harmonic)
            row[f"{prefix}_{char_name}_h{order}_real"] = float(np.real(harmonic))
            row[f"{prefix}_{char_name}_h{order}_imag"] = float(np.imag(harmonic))

rows = []
print("expected under/over pairs:", len(base_cases))

for batch_start in range(0, len(base_cases), BATCH_BASE_CASES):
    batch_base_cases = base_cases[batch_start: batch_start + BATCH_BASE_CASES]
    batch_parameters = []
    for base_case in batch_base_cases:
        for c1_offset in C1_OFFSETS:
            params = dict(base_case)
            params["C1_offset"] = c1_offset
            batch_parameters.append(params)

    probe_images, selected = run_simulation(config, batch_parameters)
    simulation_records = [dict(params) for params in batch_parameters]
    batch_pairs = select_under_over_pairs(
        simulation_records,
        COMBINATION_FIELDS,
        under_focus_c1_offset=UNDER_FOCUS_C1_OFFSET,
        over_focus_c1_offset=OVER_FOCUS_C1_OFFSET,
    )

    for local_case_index, (params, under_index, over_index) in enumerate(batch_pairs):
        stack = probe_images[:, :, [under_index, over_index]]
        profiles, coords = extract_line_profiles_from_stack(
            stack,
            num_lines=NUM_PROFILE_LINES,
            radius=PROFILE_RADIUS_PIXELS,
        )
        angles_deg = np.asarray(coords["angles_deg"], dtype=float)
        under_chars = compute_line_characteristics(profiles[:, :, 0], PROFILE_RADIUS_PIXELS)
        over_chars = compute_line_characteristics(profiles[:, :, 1], PROFILE_RADIUS_PIXELS)
        feature_values = compute_uno_values(under_chars, over_chars, angles_deg)

        row = {field: params.get(field, 0.0) for field in COMBINATION_FIELDS}
        row["case_index"] = batch_start + local_case_index
        row["under_index"] = batch_start * len(C1_OFFSETS) + under_index
        row["over_index"] = batch_start * len(C1_OFFSETS) + over_index
        row["under_C1_offset"] = UNDER_FOCUS_C1_OFFSET
        row["over_C1_offset"] = OVER_FOCUS_C1_OFFSET
        for name, value in feature_values.items():
            add_complex_columns(row, name, value)
        add_raw_harmonic_features(row, "under", under_chars, angles_deg)
        add_raw_harmonic_features(row, "over", over_chars, angles_deg)
        rows.append(row)

    del probe_images, selected, batch_parameters, simulation_records, batch_pairs
    xp.cuda.Stream.null.synchronize()
    xp.get_default_memory_pool().free_all_blocks()
    print(f"processed {min(batch_start + BATCH_BASE_CASES, len(base_cases))}/{len(base_cases)} base cases")

CSV_PATH = OUTPUT_DIR / "training_features_enhanced.csv"
with CSV_PATH.open("w", newline="") as handle:
    fieldnames = list(rows[0].keys())
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

(OUTPUT_DIR / "feature_columns_enhanced.json").write_text(json.dumps(FEATURE_COLUMNS_ENHANCED, indent=2) + "\n")
print("saved enhanced training feature table:", CSV_PATH)
print("rows:", len(rows))
for label in sorted({row["sweep_label"] for row in rows}):
    print(f"  {label}: {sum(row['sweep_label'] == label for row in rows)} rows")


## 8. Train Weighted Early-Stopping Residual Regressor

In [ ]:
def load_rows(csv_path):
    with Path(csv_path).open() as handle:
        return list(csv.DictReader(handle))

def row_float(row, name, default=0.0):
    value = row.get(name, default)
    if value in (None, ""):
        return float(default)
    return float(value)

def prepare_enhanced_dataset(csv_path, feature_columns):
    table_rows = load_rows(csv_path)
    X = np.asarray([[row_float(row, name) for name in feature_columns] for row in table_rows], dtype=np.float32)
    y = np.asarray([target_from_row(row) for row in table_rows], dtype=np.float32)
    labels = np.asarray([row.get("sweep_label", "") for row in table_rows])
    return X, y, labels, table_rows

def import_torch():
    import torch
    import torch.nn as nn
    return torch, nn

def build_enhanced_model(input_dim, output_dim, hidden_dim=128):
    torch, nn = import_torch()

    class EnhancedHybridRegressor(nn.Module):
        def __init__(self):
            super().__init__()
            self.linear = nn.Linear(input_dim, output_dim)
            self.residual = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.SiLU(),
                nn.LayerNorm(hidden_dim),
                nn.Linear(hidden_dim, hidden_dim),
                nn.SiLU(),
                nn.LayerNorm(hidden_dim),
                nn.Linear(hidden_dim, hidden_dim),
                nn.SiLU(),
                nn.Linear(hidden_dim, output_dim),
            )
            nn.init.zeros_(self.residual[-1].weight)
            nn.init.zeros_(self.residual[-1].bias)

        def forward(self, x):
            return self.linear(x) + self.residual(x)

    return EnhancedHybridRegressor()

def weighted_mse(pred, target, target_weights):
    return torch.mean((pred - target) ** 2 * target_weights[None, :])

def make_target_weights():
    # Applied after target standardization: this emphasizes the target groups
    # that were weakest in the previous run without letting raw units dominate.
    weights = {
        "C1": 1.5,
        "C3": 0.8,
        "A1_x": 1.1,
        "A1_y": 1.1,
        "B2_x": 0.8,
        "B2_y": 0.8,
        "A2_x": 0.9,
        "A2_y": 0.9,
        "S3_x": 1.5,
        "S3_y": 1.5,
        "A3_x": 1.7,
        "A3_y": 1.7,
    }
    return np.asarray([weights[name] for name in TARGET_COLUMNS], dtype=np.float32)

def describe_enhanced_model(input_dim, output_dim, hidden_dim):
    return "\n".join([
        "EnhancedHybridRegressor",
        f"  input_dim: {input_dim}",
        f"  output_dim: {output_dim}",
        "  forward(x): linear(x) + residual(x)",
        "  residual: Linear, SiLU, LayerNorm, Linear, SiLU, LayerNorm, Linear, SiLU, Linear",
        f"  hidden_dim: {hidden_dim}",
        "  final residual layer initialized to zero",
    ])

TRAIN_HIDDEN_DIM = 128
TRAIN_LEARNING_RATE = 6e-4
TRAIN_WEIGHT_DECAY = 1e-4
TRAIN_RESIDUAL_PENALTY = 1e-3
TRAIN_MAX_EPOCHS = 8000
TRAIN_EVAL_EVERY = 25
TRAIN_PATIENCE_EPOCHS = 700
TRAIN_TEST_FRACTION = 0.2
TRAIN_SEED = 7

X, y, labels, table_rows = prepare_enhanced_dataset(CSV_PATH, FEATURE_COLUMNS_ENHANCED)
train_index, test_index = train_test_split(len(X), test_fraction=TRAIN_TEST_FRACTION, seed=TRAIN_SEED)
x_scaler = Standardizer().fit(X[train_index])
y_scaler = Standardizer().fit(y[train_index])
Xn = x_scaler.transform(X).astype(np.float32)
yn = y_scaler.transform(y).astype(np.float32)

torch, nn = import_torch()
device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_enhanced_model(Xn.shape[1], yn.shape[1], hidden_dim=TRAIN_HIDDEN_DIM).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=TRAIN_LEARNING_RATE, weight_decay=TRAIN_WEIGHT_DECAY)
target_weights = torch.tensor(make_target_weights(), device=device)

x_train = torch.tensor(Xn[train_index], device=device)
y_train = torch.tensor(yn[train_index], device=device)
x_test = torch.tensor(Xn[test_index], device=device)
y_test = torch.tensor(yn[test_index], device=device)

best_test_loss = None
best_epoch = None
best_state = None
history = []
no_improve_epochs = 0

for epoch in range(TRAIN_MAX_EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    pred = model(x_train)
    residual = model.residual(x_train)
    loss = weighted_mse(pred, y_train, target_weights) + TRAIN_RESIDUAL_PENALTY * torch.mean(residual ** 2)
    loss.backward()
    optimizer.step()

    if epoch % TRAIN_EVAL_EVERY == 0 or epoch == TRAIN_MAX_EPOCHS - 1:
        model.eval()
        with torch.no_grad():
            train_loss = float(weighted_mse(model(x_train), y_train, target_weights).detach().cpu())
            test_loss = float(weighted_mse(model(x_test), y_test, target_weights).detach().cpu())
            unweighted_test_loss = float(torch.mean((model(x_test) - y_test) ** 2).detach().cpu())
        improved = best_test_loss is None or test_loss < best_test_loss - 1e-6
        if improved:
            best_test_loss = test_loss
            best_epoch = epoch
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            no_improve_epochs = 0
        else:
            no_improve_epochs += TRAIN_EVAL_EVERY
        history.append({
            "epoch": epoch,
            "train_weighted_mse_scaled": train_loss,
            "test_weighted_mse_scaled": test_loss,
            "test_mse_scaled": unweighted_test_loss,
            "best_test_weighted_mse_scaled": best_test_loss,
        })
        if epoch % 250 == 0:
            print(f"epoch={epoch} train={train_loss:.5g} test={test_loss:.5g} best={best_test_loss:.5g}")
        if no_improve_epochs >= TRAIN_PATIENCE_EPOCHS:
            print(f"early stopping at epoch {epoch}; best epoch {best_epoch}")
            break

if best_state is not None:
    model.load_state_dict({name: value.to(device) for name, value in best_state.items()})

model.eval()
with torch.no_grad():
    pred_scaled = model(torch.tensor(Xn, device=device)).detach().cpu().numpy()
pred = y_scaler.inverse_transform(pred_scaled)

print(describe_enhanced_model(Xn.shape[1], yn.shape[1], TRAIN_HIDDEN_DIM))
print("device:", device)
print("best epoch:", best_epoch)
print("best weighted test MSE:", best_test_loss)


## 9. Diagnostics and Saved Outputs

In [ ]:
def summarize_predictions(y_true, y_pred, labels, train_index, test_index):
    errors = y_pred - y_true
    abs_errors = np.abs(errors)
    metrics = {
        "n_samples": int(len(y_true)),
        "n_train": int(len(train_index)),
        "n_test": int(len(test_index)),
        "overall_mae": float(abs_errors.mean()),
        "overall_rmse": float(np.sqrt(np.mean(errors ** 2))),
        "targets": {},
        "test_targets": {},
        "labels": {},
        "training_config": {
            "max_epochs": int(TRAIN_MAX_EPOCHS),
            "eval_every": int(TRAIN_EVAL_EVERY),
            "patience_epochs": int(TRAIN_PATIENCE_EPOCHS),
            "learning_rate": float(TRAIN_LEARNING_RATE),
            "weight_decay": float(TRAIN_WEIGHT_DECAY),
            "residual_penalty": float(TRAIN_RESIDUAL_PENALTY),
            "hidden_dim": int(TRAIN_HIDDEN_DIM),
            "best_epoch": None if best_epoch is None else int(best_epoch),
            "best_test_weighted_mse_scaled": None if best_test_loss is None else float(best_test_loss),
            "feature_count": int(len(FEATURE_COLUMNS_ENHANCED)),
        },
    }
    for i, name in enumerate(TARGET_COLUMNS):
        target_errors = errors[:, i]
        metrics["targets"][name] = {
            "mae": float(np.mean(np.abs(target_errors))),
            "rmse": float(np.sqrt(np.mean(target_errors ** 2))),
            "p95_abs_error": float(np.quantile(np.abs(target_errors), 0.95)),
        }
        test_errors = errors[test_index, i]
        metrics["test_targets"][name] = {
            "mae": float(np.mean(np.abs(test_errors))),
            "rmse": float(np.sqrt(np.mean(test_errors ** 2))),
            "p95_abs_error": float(np.quantile(np.abs(test_errors), 0.95)),
        }
    for label in sorted(set(labels)):
        for split_name, indices in [("all", np.arange(len(labels))), ("test", test_index)]:
            mask_indices = indices[labels[indices] == label]
            if len(mask_indices) == 0:
                continue
            label_errors = errors[mask_indices]
            metrics["labels"].setdefault(label, {})[split_name] = {
                "n": int(len(mask_indices)),
                "mae": float(np.mean(np.abs(label_errors))),
                "rmse": float(np.sqrt(np.mean(label_errors ** 2))),
            }
    return metrics

def write_history_csv(path, history):
    with Path(path).open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(history[0].keys()))
        writer.writeheader()
        writer.writerows(history)

def write_predictions_csv(path, table_rows, y_true, y_pred, train_index, test_index):
    split = np.asarray(["train"] * len(table_rows), dtype=object)
    split[test_index] = "test"
    with Path(path).open("w", newline="") as handle:
        fieldnames = ["row_index", "split", "sweep_label"]
        fieldnames += [f"true_{name}" for name in TARGET_COLUMNS]
        fieldnames += [f"pred_{name}" for name in TARGET_COLUMNS]
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for index, row in enumerate(table_rows):
            out = {"row_index": index, "split": split[index], "sweep_label": row.get("sweep_label", "")}
            out.update({f"true_{name}": float(y_true[index, i]) for i, name in enumerate(TARGET_COLUMNS)})
            out.update({f"pred_{name}": float(y_pred[index, i]) for i, name in enumerate(TARGET_COLUMNS)})
            writer.writerow(out)

def plot_training_history(history, output_dir):
    output_dir = Path(output_dir)
    epochs = [item["epoch"] for item in history]
    train = [item["train_weighted_mse_scaled"] for item in history]
    test = [item["test_weighted_mse_scaled"] for item in history]
    fig, axis = plt.subplots(figsize=(7.0, 4.6))
    axis.plot(epochs, train, label="train weighted")
    axis.plot(epochs, test, label="test weighted")
    if best_epoch is not None:
        axis.axvline(best_epoch, color="black", linestyle="--", linewidth=1, label=f"best {best_epoch}")
    axis.set_yscale("log")
    axis.set_xlabel("epoch")
    axis.set_ylabel("scaled weighted MSE")
    axis.set_title("Enhanced feature-regression training")
    axis.grid(alpha=0.3)
    axis.legend()
    fig.tight_layout()
    path = output_dir / "training_history_enhanced.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    return path

def plot_test_scatter_by_label(y_true, y_pred, labels, test_index, output_dir):
    output_dir = Path(output_dir)
    test_labels = labels[test_index]
    unique_labels = sorted(set(test_labels))
    cmap = plt.get_cmap("tab20", len(unique_labels))
    color_map = {label: cmap(i) for i, label in enumerate(unique_labels)}
    fig, axes = plt.subplots(3, 4, figsize=(17, 12), squeeze=False)
    for i, name in enumerate(TARGET_COLUMNS):
        axis = axes[i // 4, i % 4]
        for label in unique_labels:
            label_indices = test_index[test_labels == label]
            axis.scatter(
                y_true[label_indices, i],
                y_pred[label_indices, i],
                s=10,
                alpha=0.55,
                color=color_map[label],
                label=label if i == 0 else None,
            )
        values = np.concatenate([y_true[test_index, i], y_pred[test_index, i]])
        lo = float(np.nanmin(values))
        hi = float(np.nanmax(values))
        if lo == hi:
            hi = lo + 1
        axis.plot([lo, hi], [lo, hi], color="black", linestyle="--", linewidth=1)
        axis.set_title(name)
        axis.set_xlabel("true")
        axis.set_ylabel("pred")
        axis.grid(alpha=0.25)
    handles, labels_for_legend = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels_for_legend, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
    fig.suptitle("Test split predicted vs true targets, colored by sweep label", fontsize=14)
    fig.tight_layout(rect=[0, 0, 0.86, 0.97])
    path = output_dir / "prediction_scatter_test_by_label.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    return path

def harmonic_amplitude_phase_metrics(y_true, y_pred, indices):
    rows_out = []
    target_lookup = {name: i for i, name in enumerate(TARGET_COLUMNS)}
    for name, amp_field, phase_field, order in HARMONIC_TARGETS:
        ix = target_lookup[f"{name}_x"]
        iy = target_lookup[f"{name}_y"]
        tx = y_true[indices, ix]
        ty = y_true[indices, iy]
        px = y_pred[indices, ix]
        py = y_pred[indices, iy]
        true_amp = np.hypot(tx, ty)
        pred_amp = np.hypot(px, py)
        vector_error = np.hypot(px - tx, py - ty)
        nonzero = true_amp > 1e-9
        period = 360.0 / order
        true_phase = (np.degrees(np.arctan2(ty[nonzero], tx[nonzero])) / order) % period
        pred_phase = (np.degrees(np.arctan2(py[nonzero], px[nonzero])) / order) % period
        phase_error = np.abs((pred_phase - true_phase + period / 2) % period - period / 2)
        rows_out.append({
            "target": name,
            "n": int(len(indices)),
            "amp_mae": float(np.mean(np.abs(pred_amp - true_amp))),
            "vector_rmse": float(np.sqrt(np.mean(vector_error ** 2))),
            "phase_median_deg": float(np.median(phase_error)) if len(phase_error) else None,
            "phase_p95_deg": float(np.quantile(phase_error, 0.95)) if len(phase_error) else None,
        })
    return rows_out

def write_amplitude_bin_metrics(path, y_true, y_pred, labels, test_index):
    target_lookup = {name: i for i, name in enumerate(TARGET_COLUMNS)}
    out = []
    for name, amp_field, phase_field, order in HARMONIC_TARGETS:
        ix = target_lookup[f"{name}_x"]
        iy = target_lookup[f"{name}_y"]
        true_amp = np.hypot(y_true[test_index, ix], y_true[test_index, iy])
        vector_error = np.hypot(y_pred[test_index, ix] - y_true[test_index, ix], y_pred[test_index, iy] - y_true[test_index, iy])
        edges = np.quantile(true_amp, [0, 0.25, 0.5, 0.75, 1.0])
        edges = np.unique(edges)
        for lo, hi in zip(edges[:-1], edges[1:]):
            mask = (true_amp >= lo) & (true_amp <= hi if hi == edges[-1] else true_amp < hi)
            if not np.any(mask):
                continue
            out.append({
                "target": name,
                "amp_bin_low": float(lo),
                "amp_bin_high": float(hi),
                "n": int(np.sum(mask)),
                "vector_mae": float(np.mean(vector_error[mask])),
                "vector_rmse": float(np.sqrt(np.mean(vector_error[mask] ** 2))),
            })
    with Path(path).open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(out[0].keys()))
        writer.writeheader()
        writer.writerows(out)
    return out


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def git_commit():
    try:
        result = subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"],
            cwd=ROOT,
            check=True,
            capture_output=True,
            text=True,
        )
    except Exception:
        return None
    return result.stdout.strip()

def write_enhanced_run_tracking():
    created_utc = datetime.now(timezone.utc).isoformat()
    run_config = {
        "run_name": RUN_NAME,
        "random_seed": RANDOM_SEED,
        "random_coupled_cases": RANDOM_COUPLED_CASES,
        "random_case_counts": RANDOM_CASE_COUNTS,
        "batch_base_cases": BATCH_BASE_CASES,
        "profile_radius_pixels": PROFILE_RADIUS_PIXELS,
        "profile_step_degrees": PROFILE_STEP_DEGREES,
        "feature_count": len(FEATURE_COLUMNS_ENHANCED),
        "train_hidden_dim": TRAIN_HIDDEN_DIM,
        "train_learning_rate": TRAIN_LEARNING_RATE,
        "train_weight_decay": TRAIN_WEIGHT_DECAY,
        "train_residual_penalty": TRAIN_RESIDUAL_PENALTY,
        "train_max_epochs": TRAIN_MAX_EPOCHS,
        "train_patience_epochs": TRAIN_PATIENCE_EPOCHS,
        "train_test_fraction": TRAIN_TEST_FRACTION,
        "train_seed": TRAIN_SEED,
    }
    manifest = {
        "run_name": RUN_NAME,
        "created_utc": created_utc,
        "git_commit": git_commit(),
        "python": sys.version,
        "platform": platform.platform(),
        "device": device,
        "dataset": {
            "csv_path": str(CSV_PATH),
            "csv_sha256": file_sha256(CSV_PATH),
            "n_rows": len(table_rows),
            "n_train": len(train_index),
            "n_test": len(test_index),
            "label_counts": dict(Counter(labels.tolist())),
        },
        "model": {
            "type": "EnhancedHybridRegressor",
            "input_dim": len(FEATURE_COLUMNS_ENHANCED),
            "output_dim": len(TARGET_COLUMNS),
            "hidden_dim": TRAIN_HIDDEN_DIM,
            "summary": describe_enhanced_model(Xn.shape[1], yn.shape[1], TRAIN_HIDDEN_DIM),
        },
        "training": metrics["training_config"],
        "feature_columns": FEATURE_COLUMNS_ENHANCED,
        "target_columns": TARGET_COLUMNS,
        "extra_config": run_config,
    }
    (OUTPUT_DIR / "run_manifest_enhanced.json").write_text(json.dumps(manifest, indent=2) + "\n")

    registry_path = OUTPUT_ROOT / "model_registry_enhanced.csv"
    row = {
        "created_utc": created_utc,
        "run_name": RUN_NAME,
        "output_dir": str(OUTPUT_DIR),
        "git_commit": manifest["git_commit"],
        "n_rows": len(table_rows),
        "n_train": len(train_index),
        "n_test": len(test_index),
        "input_dim": len(FEATURE_COLUMNS_ENHANCED),
        "hidden_dim": TRAIN_HIDDEN_DIM,
        "max_epochs": TRAIN_MAX_EPOCHS,
        "best_epoch": metrics["training_config"].get("best_epoch"),
        "best_test_weighted_mse_scaled": metrics["training_config"].get("best_test_weighted_mse_scaled"),
        "overall_mae": metrics.get("overall_mae"),
        "overall_rmse": metrics.get("overall_rmse"),
        "test_C1_rmse": metrics.get("test_targets", {}).get("C1", {}).get("rmse"),
        "test_S3_x_rmse": metrics.get("test_targets", {}).get("S3_x", {}).get("rmse"),
        "test_S3_y_rmse": metrics.get("test_targets", {}).get("S3_y", {}).get("rmse"),
        "test_A3_x_rmse": metrics.get("test_targets", {}).get("A3_x", {}).get("rmse"),
        "test_A3_y_rmse": metrics.get("test_targets", {}).get("A3_y", {}).get("rmse"),
    }
    write_header = not registry_path.exists()
    with registry_path.open("a", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(row))
        if write_header:
            writer.writeheader()
        writer.writerow(row)

metrics = summarize_predictions(y, pred, labels, train_index, test_index)
metrics["harmonic_test"] = harmonic_amplitude_phase_metrics(y, pred, test_index)

(OUTPUT_DIR / "metrics_enhanced.json").write_text(json.dumps(metrics, indent=2) + "\n")
(OUTPUT_DIR / "normalization_enhanced.json").write_text(json.dumps({
    "features": FEATURE_COLUMNS_ENHANCED,
    "targets": TARGET_COLUMNS,
    "x": x_scaler.to_dict(),
    "y": y_scaler.to_dict(),
    "target_weights": {name: float(value) for name, value in zip(TARGET_COLUMNS, make_target_weights())},
}, indent=2) + "\n")
(OUTPUT_DIR / "model_summary_enhanced.txt").write_text(describe_enhanced_model(Xn.shape[1], yn.shape[1], TRAIN_HIDDEN_DIM) + "\n")
write_history_csv(OUTPUT_DIR / "history_enhanced.csv", history)
write_predictions_csv(OUTPUT_DIR / "predictions_enhanced.csv", table_rows, y, pred, train_index, test_index)
write_amplitude_bin_metrics(OUTPUT_DIR / "amplitude_bin_metrics_enhanced.csv", y, pred, labels, test_index)
plot_training_history(history, OUTPUT_DIR)
plot_test_scatter_by_label(y, pred, labels, test_index, OUTPUT_DIR)
write_enhanced_run_tracking()

torch.save({
    "model_state_dict": model.state_dict(),
    "feature_columns": FEATURE_COLUMNS_ENHANCED,
    "target_columns": TARGET_COLUMNS,
    "device_used": device,
    "training_config": metrics["training_config"],
}, OUTPUT_DIR / "enhanced_feature_regressor.pt")

print("overall test target metrics:")
for name, item in metrics["test_targets"].items():
    print(f"  {name}: MAE={item['mae']:.4g}, RMSE={item['rmse']:.4g}, p95={item['p95_abs_error']:.4g}")
print("harmonic test metrics:")
for item in metrics["harmonic_test"]:
    print(item)


## 10. Save and Download Results

In [ ]:
ZIP_PATH = OUTPUT_DIR / f"{RUN_NAME}_feature_regression_enhanced_results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path == ZIP_PATH or path.is_dir():
            continue
        zf.write(path, path.name)
    registry_path = OUTPUT_ROOT / "model_registry_enhanced.csv"
    if registry_path.exists():
        zf.write(registry_path, registry_path.name)
print("saved:", ZIP_PATH)

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except Exception as exc:
    print("Browser download skipped or unsupported:", exc)


## 11. Optional: Mirror Results to Google Drive

In [ ]:
# Set this to True when you want Colab to mount Google Drive and copy results there.
MIRROR_TO_DRIVE = False
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/aberration_simulation/feature_regression_enhanced")
DRIVE_OUTPUT_DIR = DRIVE_OUTPUT_ROOT / RUN_NAME

if MIRROR_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            shutil.copy2(path, DRIVE_OUTPUT_DIR / path.name)
    registry_path = OUTPUT_ROOT / "model_registry_enhanced.csv"
    if registry_path.exists():
        DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
        shutil.copy2(registry_path, DRIVE_OUTPUT_ROOT / registry_path.name)
    print("mirrored enhanced outputs to:", DRIVE_OUTPUT_DIR)
else:
    print("Drive mirroring skipped.")
